In [1]:
%pip install lightgbm xgboost catboost


  Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl (3.6 MB)
  Using cached xgboost-3.0.2-py3-none-manylinux_2_28_x86_64.whl (253.9 MB)
  Using cached catboost-1.2.8-cp310-cp310-manylinux2014_x86_64.whl (99.2 MB)
  Using cached nvidia_nccl_cu12-2.26.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (318.1 MB)
  Using cached graphviz-0.20.3-py3-none-any.whl (47 kB)
  Using cached plotly-6.1.2-py3-none-any.whl (16.3 MB)
  Using cached narwhals-1.41.0-py3-none-any.whl (357 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import joblib
from datetime import datetime
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from src.config import *
from src.utils import get_latest_file
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

# Charger les données
dataset_path = get_latest_file(DATA_FINAL_CLEANED_DATASET_DIR)
df = pd.read_csv(dataset_path)

# Préparation des données
target = 'IS_WIN'
drop_cols = ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID', 'SEASON', 'GAME_DATE']
features = [col for col in df.columns if col not in drop_cols + [target]]
df = df.dropna(subset=features)

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Définis tes modèles de base
estimators = [
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('lgbm', LGBMClassifier(n_estimators=150, num_leaves=64, random_state=42, n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.7, colsample_bytree=0.7, random_state=42, eval_metric='logloss', n_jobs=-1, use_label_encoder=False)),
    ('cat', CatBoostClassifier(n_estimators=200, learning_rate=0.05, depth=6, rsm=0.8, verbose=0, random_state=42)),
    ('hgb', HistGradientBoostingClassifier(max_iter=200, random_state=42)),
    ('lr', make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, solver='liblinear', penalty='l2'))),
    ('mlp', make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42))),
    ('et', ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('knn', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15, n_jobs=-1)))
]

# Meta-model (peut être LogisticRegression, simple et efficace)
meta_model = LogisticRegression(solver='lbfgs', max_iter=5000)

# Création du stacking
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model,
    passthrough=False,
    cv=5,
    n_jobs=-1,
    verbose=2
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', stack)
])

# Entraînement
pipeline.fit(X_train, y_train)

# Évaluation
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))

# Sauvegarde
today = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

os.makedirs(DATA_MODELS_DIR, exist_ok=True)

stacking_model_path = os.path.join(DATA_MODELS_DIR, f"stacking_model_{today}.joblib")
joblib.dump(pipeline,stacking_model_path)
print(f"Model saved in {stacking_model_path}")


/opt/conda/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [00:54:42] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[00:56:54] WARNING: /workspace/src/le

[LightGBM] [Info] Number of positive: 12110, number of negative: 12198
[LightGBM] [Info] Number of positive: 12109, number of negative: 12199
[LightGBM] [Info] Number of positive: 12110, number of negative: 12198
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.206744 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Number of positive: 12109, number of negative: 12199
[LightGBM] [Info] Total Bins 43756
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.205786 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 43768
[LightGBM] [Info] Number of positive: 12110, number of negative: 12198
[LightGBM] [Info] Number of data points in the train set: 24308, number of used features: 206
[LightGBM] [Info] Number of data points in the train set: 24308, number of used features: 206

[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   37.9s remaining:   25.3s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   40.2s finished
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   43.9s remaining:   29.3s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   51.1s finished
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  2.6min remaining:  1.7min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  2.6min remaining:  1.7min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  2.6min finished
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  3.0min finished
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  4.1min remaining:  2.7min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  4.1min finished


[LightGBM] [Info] Number of positive: 15137, number of negative: 15248
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.315821 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 43822
[LightGBM] [Info] Number of data points in the train set: 30385, number of used features: 206
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.498173 -> initscore=-0.007306
[LightGBM] [Info] Start training from score -0.007306


[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  4.4min remaining:  3.0min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  4.6min finished
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  5.1min remaining:  3.4min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  5.3min finished
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  6.0min remaining:  4.0min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  6.0min finished
/opt/conda/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:702: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:702: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed: 11.0min remaining:  7.3min
/opt/

ROC AUC: 0.7433894057466598
Model saved in data/models/stacking_model_2025-05-30_01-07-53.joblib
